# Practical Exam: Hotel Operations

LuxurStay Hotels is a major, international chain of hotels. They offer hotels for both business and leisure travellers in major cities across the world. The chain prides themselves on the level of customer service that they offer. 

However, the management has been receiving complaints about slow room service in some hotel branches. As these complaints are impacting the customer satisfaction rates, it has become a serious issue. Recent data shows that customer satisfaction has dropped from the 4.5 rating that they expect. 

You are working with the Head of Operations to identify possible causes and hotel branches with the worst problems. 

## Data

The following schema diagram shows the tables available. You have only been provided with data where customers provided a feedback rating.

![hotel_operations](hotel_operations.png)

In [7]:
import pandas as pd

df = pd.read_csv('./exported/public.branch.csv')

print(df.head(100))
df.describe()
df.info()
df.isnull().sum()

    index   id location  total_rooms  staff_count opening_date target_guests
0       0    1    LATAM        168.0          178         2017      Business
1       1    2     APAC        154.0           82         2010       Leisure
2       2    3     APAC        212.0          467         2003       Leisure
3       3    4     APAC        230.0          387            -      Business
4       4    5     APAC        292.0          293         2002      Business
..    ...  ...      ...          ...          ...          ...           ...
95     95   96     APAC        237.0          257         2000      Business
96     96   97     APAC        107.0          169         2005      Business
97     97   98     EMEA        196.0          126         2002       Leisure
98     98   99     APAC        242.0          251         2021      Business
99     99  100    LATAM        349.0          612         2020      Business

[100 rows x 7 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10

index             0
id                0
location         23
total_rooms      10
staff_count       0
opening_date      0
target_guests     0
dtype: int64

In [10]:
import pandas as pd

df = pd.read_csv('./exported/public.request.csv')

print(df.head(17682))
df.describe()
df.info()
df.isnull().sum()

       Unnamed: 0     id  service_id  branch_id  time_taken  request_time  \
0               0      1           3         92           7  Day-Off-Peak   
1               1      2           1         43           2  Day-Off-Peak   
2               2      3           2         63          13      Day-Peak   
3               3      4           3         89           7      Day-Peak   
4               4      5           4        100           9      Day-Peak   
...           ...    ...         ...        ...         ...           ...   
17677       17677  17678           3         68           7      Day-Peak   
17678       17678  17679           4         67           9  Day-Off-Peak   
17679       17679  17680           1          9           4      Day-Peak   
17680       17680  17681           1         23           2         Night   
17681       17681  17682           1         41           1         Night   

       rating  
0           4  
1           4  
2           4  
3          

Unnamed: 0      0
id              0
service_id      0
branch_id       0
time_taken      0
request_time    0
rating          0
dtype: int64

In [11]:
import pandas as pd

df = pd.read_csv('./exported/public.service.csv')

print(df.head(100))
df.describe()
df.info()
df.isnull().sum()

   index  id description
0      0   1        Meal
1      1   2     Laundry
2      2   3    Cleaning
3      3   4       Other
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   index        4 non-null      int64 
 1   id           4 non-null      int64 
 2   description  4 non-null      object
dtypes: int64(2), object(1)
memory usage: 228.0+ bytes


index          0
id             0
description    0
dtype: int64

---

# Task 1

Before you can start any analysis, you need to confirm that the data is accurate and reflects what you expect to see. 

It is known that there are some issues with the `branch` table, and the data team have provided the following data description. 

Write a query to return data matching this description, including identifying and cleaning all invalid values. You must match all column names and description criteria. Your output should be a DataFrame named 'clean_branch_data'.

| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
|id | Nominal. The unique identifier of the hotel. </br>Missing values are not possible due to the database structure.|
| location | Nominal. The location of the particular hotel. One of four possible values, 'EMEA', 'NA', 'LATAM' and 'APAC'. </br>Missing values should be replaced with “Unknown”. |
| total_rooms | Discrete. The total number of rooms in the hotel. Must be a positive integer between 1 and 400. </br>Missing values should be replaced with the default number of rooms, 100. |
| staff_count | Discrete. The number of staff employeed in the hotel service department. </br>Missing values should be replaced with the total_rooms multiplied by 1.5. |
| opening_date | Discrete. The year in which the hotel opened. This can be any value between 2000 and 2023. </br>Missing values should be replaced with 2023. |
| target_guests | Nominal. The primary type of guest that is expected to use the hotel. Can be one of 'Leisure' or 'Business'. </br>Missing values should be replaced with 'Leisure'. |

Query:

```sql
SELECT 
    id,
    CASE
        WHEN location IN ('EMEA', 'NA', 'LATAM', 'APAC') THEN location
        ELSE 'Unknown'
    END AS location,
    CASE
        WHEN total_rooms IS NULL OR total_rooms NOT BETWEEN 1 AND 400 THEN 100
        ELSE total_rooms
    END AS total_rooms,
    COALESCE(
        staff_count, 
        (CASE 
            WHEN total_rooms IS NULL OR total_rooms NOT BETWEEN 1 AND 400 THEN 100
            ELSE total_rooms
        END) * 1.5
    ) AS staff_count,
    CASE
        WHEN opening_date IS NULL OR opening_date = '-' OR opening_date NOT BETWEEN '2000' AND '2023' THEN '2023'
        ELSE opening_date
    END AS opening_date,
    CASE
        WHEN target_guests IN ('Leisure', 'Business') THEN target_guests
        WHEN LOWER(target_guests) LIKE 'b%' THEN 'Business'
        ELSE 'Leisure'
    END AS target_guests
FROM public.branch;
```

### Result

In [8]:
import pandas as pd

df = pd.read_csv('./exported/task1_export.csv')

print(df.head(100))

    index   id location  total_rooms  staff_count  opening_date target_guests
0       0    1    LATAM          168          178          2017      Business
1       1    2     APAC          154           82          2010       Leisure
2       2    3     APAC          212          467          2003       Leisure
3       3    4     APAC          230          387          2023      Business
4       4    5     APAC          292          293          2002      Business
..    ...  ...      ...          ...          ...           ...           ...
95     95   96     APAC          237          257          2000      Business
96     96   97     APAC          107          169          2005      Business
97     97   98     EMEA          196          126          2002       Leisure
98     98   99     APAC          242          251          2021      Business
99     99  100    LATAM          349          612          2020      Business

[100 rows x 7 columns]


---

# Task 2

The Head of Operations wants to know whether there is a difference in time taken to respond to a customer request in each hotel. They already know that different services take different lengths of time. 

Calculate the average and maximum duration for each branch and service. 
- Your output should be a DataFrame named 'average_time_service'
- It should include the columns `service_id`, `branch_id`, `avg_time_taken` and `max_time_taken`
- Values should be rounded to two decimal places where appropriate. 

Query:

```sql
SELECT service_id, 
		branch_id, 
		ROUND(AVG(time_taken), 2) AS avg_time_taken, 
		MAX(time_taken) AS max_time_taken
FROM public.request
GROUP BY service_id, branch_id;
```

### Result

In [13]:
import pandas as pd

df = pd.read_csv('./exported/task2_export.csv')

print(df.head(385))

     index  service_id  branch_id  avg_time_taken  max_time_taken
0        0           2         46           13.09              16
1        1           4         99            9.13              13
2        2           1          8            2.56              10
3        3           2         13           13.53              17
4        4           1         46            2.08               4
..     ...         ...        ...             ...             ...
380    380           4         73            9.43              13
381    381           4         88            9.36              12
382    382           1         89            2.77               7
383    383           4         31            9.00               9
384    384           4         72            9.14              11

[385 rows x 5 columns]


---

# Task 3

The management team want to target improvements in `Meal` and `Laundry` service in Europe (`EMEA`) and Latin America (`LATAM`). 

Write a query to return the `description` of the service, the `id` and `location` of the branch, the id of the request as `request_id` and the `rating` for the services and locations of interest to the management team. 

Your output should be a DataFrame named 'target_hotels'.

Use the original branch table, not the output of task 1. 

Query:

```sql
SELECT
    s.description AS service_description,
    b.id AS branch_id,
    b.location,
    r.id AS request_id,
    r.rating
FROM
    request r
    JOIN branch b ON b.id = r.branch_id
    JOIN service s ON r.service_id = s.id
WHERE
    s.description IN ('Meal', 'Laundry')
    AND b.location IN ('EMEA', 'LATAM');
```

### Result


In [14]:
import pandas as pd

df = pd.read_csv('./exported/task3_export.csv')

print(df.head(5047))

      index service_description  branch_id location  request_id  rating
0         0             Laundry         63     EMEA           3       4
1         1             Laundry         69    LATAM           6       5
2         2                Meal         44     EMEA          18       4
3         3             Laundry         57    LATAM          19       3
4         4                Meal          1    LATAM          21       4
...     ...                 ...        ...      ...         ...     ...
5042   5042                Meal         30     EMEA       17662       4
5043   5043                Meal         64    LATAM       17669       4
5044   5044                Meal         51    LATAM       17674       5
5045   5045                Meal         23     EMEA       17681       5
5046   5046                Meal         41    LATAM       17682       4

[5047 rows x 6 columns]


---

# Task 4

So that you can take a more detailed look at the lowest performing hotels, you want to get service and branch information where the average rating for the branch and service combination is lower than 4.5 - the target set by management.  

- Your output should be a DataFrame named 'average_rating'
- It should return the `service_id` and `branch_id`, and the average rating (`avg_rating`)
- Values should be rounded to 2 decimal places where appropriate.

Query:

```sql
SELECT service_id, branch_id, ROUND(AVG(rating),2) AS avg_rating
FROM public.request
GROUP BY service_id, branch_id
HAVING AVG(rating) < 4.5;
```

### Result

In [15]:
import pandas as pd

df = pd.read_csv('./exported/task4_export.csv')

print(df.head(5047))

     index  service_id  branch_id  avg_rating
0        0           2         46        3.78
1        1           4         99        3.83
2        2           1          8        3.64
3        3           1         46        3.81
4        4           3         15        4.00
..     ...         ...        ...         ...
210    210           3          8        3.38
211    211           1         64        3.59
212    212           4         93        3.72
213    213           4         88        3.60
214    214           4         31        4.00

[215 rows x 4 columns]
